In [28]:
import os
import base64
from io import BytesIO
from PIL import Image
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [29]:
# Inicialización

load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key sin configurar")

MODEL = "gpt-4o-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [30]:
system_message = "Eres un asistente útil y cortéz para hacer traducciones solo del español al inglés. "
system_message += "Debes responder siempre en espanol a excepcion cuando debas traducir al inglés."
system_message += "Se siempre preciso. Si te solicitan traducción a un idioma distinto al inglés."
system_message += "Debes responder que no estás facultado en ese idioma."

In [31]:
# Función para generar la imagen con DALL·E
def generar_imagen(prompt_traduccion):
    image_response = openai.images.generate(
        model="dall-e-3",
        prompt=f"Representación visual de: {prompt_traduccion}, en estilo digital artístico moderno.",
        size="512x512",
        n=1,
        response_format="b64_json",
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [32]:
def chat_y_arte(message, history):
    # Construir mensajes para la API
    messages = [{"role": "system", "content": system_message}] + history
    messages.append({"role": "user", "content": str(message)})

    # Validación final (opcional pero útil para debugging)
    for m in messages:
        assert isinstance(m, dict) and "role" in m and "content" in m

    # Llamada al modelo
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    traduccion = response.choices[0].message.content

    # Generar imagen solo si la traducción parece válida
    if any(char.isascii() and char.isalpha() for char in traduccion):
        try:
            imagen = generar_imagen(traduccion)
        except Exception as e:
            print(f"❌ Error generando imagen: {e}")
            imagen = None
    else:
        imagen = None

    return traduccion, imagen



In [33]:
# Tema personalizado
tema_personal = gr.themes.Base(primary_hue="gray", font=["Arial", "sans-serif"]).set(
    body_text_color="gray",
    background_fill_primary="black",
    input_background_fill="white",
    block_background_fill="black"
)

# Interfaz Gradio extendida
with gr.Blocks(theme=tema_personal) as interfaz:
    gr.Markdown("## 🧠 Traductor de Chat IA al idioma Inglés + Imagen AI 🎨")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(type='messages')
            state = gr.State([])  # historial de conversación
            msg = gr.Textbox(placeholder="Dime la oración a traducir", label="Mensaje")
            enviar = gr.Button("Enviar")

        with gr.Column(scale=2):
            imagen_output = gr.Image(label="Imagen generada")

    def procesar(message, history):
        output, imagen = chat_y_arte(message, history)
        history.append((message, output))
        return history, history, imagen

    enviar.click(procesar, inputs=[msg, state], outputs=[chatbot, state, imagen_output])
    msg.submit(procesar, inputs=[msg, state], outputs=[chatbot, state, imagen_output])

interfaz.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "C:\Users\gfern\Documents\A_Frogames_LLMs\Proyectos\llm_engineering_curso\llms\Lib\site-packages\gradio\queueing.py", line 745, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\gfern\Documents\A_Frogames_LLMs\Proyectos\llm_engineering_curso\llms\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\gfern\Documents\A_Frogames_LLMs\Proyectos\llm_engineering_curso\llms\Lib\site-packages\gradio\blocks.py", line 2127, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\gfern\Documents\A_Frogames_LLMs\Proyectos\llm_engineering_curso\llms\Lib\site-packages\gradio\blocks.py", line 1904, in postprocess_